# Generate Branded AGOL Thumbnails

Stamp a consistent branded thumbnail (item **type** in the header, item **title** in the footer) onto every ArcGIS Online item you own, from one PNG template. Text **auto-fits** each band by measured pixel width.

**How to use:** edit the **CONFIG** cell, then *Run All*. Leave `DRY_RUN = True` for the first pass (renders PNGs locally, touches nothing), review the output folder, then set `DRY_RUN = False` to write the thumbnails to your portal.

**Make it yours:** the geometry constants (`HEADER_BOX`, `FOOTER_BOX`, the font auto-fit ranges) live in the *render core* cell -- edit them to match your own template. Adapted from an All Things Spatial workflow.

## 1 - Config

In [ ]:
# ============================================================================
# CONFIG -- edit these, then Run All. Nothing else needs changing for a basic run.
# ============================================================================
TEMPLATE_PATH   = "ats-base-template.png"   # your 600x400 PNG template
OUTPUT_DIR      = "thumbnail_uploads"        # local folder for rendered PNGs
SCOPE           = "entire"                   # "entire"  OR  "selected"
FOLDERS         = []                          # used when SCOPE="selected"; "(root)" = root
ITEM_TYPE       = None                         # e.g. "Feature Service"; None = all types
MAX_TITLE_CHARS = 50                           # pre-cap before ellipsis
MAX_TITLE_LINES = 3                            # footer title line budget
BRAND_BLUE      = "#1C77FF"                    # header (item type) color
MAX_ITEMS       = 2000                         # safety cap
DRY_RUN         = True                         # True = render locally only; flip to write to AGOL


## 2 - Render core (edit geometry here to fit your template)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# ---- template geometry (px). Edit these to match YOUR template ----------
HEADER_BOX = {"x": 95, "y": 14, "w": 410, "h": 110}   # item TYPE zone  (centered)
FOOTER_BOX = {"x": 24, "y": 296, "w": 338, "h": 98}   # item TITLE zone (left-aligned)

# Font auto-fit ranges (px): Hi = biggest tried; Lo = smallest before ellipsis.
TYPE_SIZE_HI,  TYPE_SIZE_LO  = 54, 30
TITLE_SIZE_HI, TITLE_SIZE_LO = 40, 20
LINE_SPACING = 1.06                  # multiplier on the font's line height
DEFAULT_BRAND_BLUE = "#1C77FF"

# First existing bold TrueType wins (Windows / ArcGIS Notebooks / macOS / Linux).
FONT_CANDIDATES = [
    "arialbd.ttf",
    "C:/Windows/Fonts/arialbd.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/Library/Fonts/Arial Bold.ttf",
    "DejaVuSans-Bold.ttf",
]


def resolve_font_path():
    """Return the first usable bold TrueType font path."""
    for cand in FONT_CANDIDATES:
        try:
            ImageFont.truetype(cand, 24)
            return cand
        except OSError:
            continue
    raise OSError("No bold TrueType font found -- add one to FONT_CANDIDATES.")


def _cap_chars(text, max_chars):
    """Pre-cap an absurdly long title at a word boundary <= max_chars, + ellipsis."""
    text = " ".join(text.split())
    if max_chars and len(text) > max_chars:
        clipped = text[:max_chars].rsplit(" ", 1)[0].rstrip()
        text = (clipped or text[:max_chars]) + "…"
    return text


def _wrap_to_width(draw, text, font, box_w):
    """Greedy word-wrap by MEASURED pixel width; hard-splits a word wider than the box."""
    words = text.split()
    lines, cur = [], ""
    for word in words:
        trial = word if not cur else cur + " " + word
        if draw.textlength(trial, font=font) <= box_w:
            cur = trial
            continue
        if cur:
            lines.append(cur); cur = ""
        if draw.textlength(word, font=font) > box_w:
            piece = ""
            for ch in word:
                if draw.textlength(piece + ch, font=font) <= box_w:
                    piece += ch
                else:
                    if piece:
                        lines.append(piece)
                    piece = ch
            cur = piece
        else:
            cur = word
    if cur:
        lines.append(cur)
    return lines


def _line_height(font):
    asc, desc = font.getmetrics()
    return int((asc + desc) * LINE_SPACING)


def fit_text_block(draw, text, font_path, box, max_lines, size_hi, size_lo):
    """Largest font size at which text wraps inside box (width, height, max_lines).
    If even size_lo overflows, wrap at size_lo, clip to max_lines, ellipsize last line."""
    for size in range(size_hi, size_lo - 1, -1):
        font = ImageFont.truetype(font_path, size)
        lines = _wrap_to_width(draw, text, font, box["w"])
        lh = _line_height(font)
        if len(lines) <= max_lines and len(lines) * lh <= box["h"]:
            return font, lines, lh
    font = ImageFont.truetype(font_path, size_lo)
    lh = _line_height(font)
    max_lines = max(1, min(max_lines, box["h"] // lh))
    lines = _wrap_to_width(draw, text, font, box["w"])
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        last = lines[-1]
        while last and draw.textlength(last + "…", font=font) > box["w"]:
            last = last[:-1].rstrip()
        lines[-1] = last + "…"
    return font, lines, lh


def render_thumbnail(template_path, out_dir, title, item_type, item_id,
                     brand_blue=DEFAULT_BRAND_BLUE, max_title_chars=50,
                     max_title_lines=3, font_path=None):
    """Stamp item TYPE (header, centered) + TITLE (footer, left) onto the template."""
    template_path = Path(template_path); out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    font_path = font_path or resolve_font_path()
    img = Image.open(template_path).convert("RGBA")
    draw = ImageDraw.Draw(img)

    type_text = " ".join((item_type or "").upper().split())
    if type_text:
        tfont, tlines, tlh = fit_text_block(draw, type_text, font_path, HEADER_BOX,
                                            2, TYPE_SIZE_HI, TYPE_SIZE_LO)
        y = HEADER_BOX["y"] + max(0, (HEADER_BOX["h"] - len(tlines) * tlh) // 2)
        for line in tlines:
            w = draw.textlength(line, font=tfont)
            x = HEADER_BOX["x"] + (HEADER_BOX["w"] - w) / 2
            draw.text((x, y), line, font=tfont, fill=brand_blue); y += tlh

    title_text = _cap_chars(title or "", max_title_chars)
    if title_text:
        ffont, flines, flh = fit_text_block(draw, title_text, font_path, FOOTER_BOX,
                                            max_title_lines, TITLE_SIZE_HI, TITLE_SIZE_LO)
        y = FOOTER_BOX["y"] + max(0, (FOOTER_BOX["h"] - len(flines) * flh) // 2)
        for line in flines:
            draw.text((FOOTER_BOX["x"], y), line, font=ffont, fill="white"); y += flh

    out_path = out_dir / "{}_thumbnail.png".format(item_id)
    img.save(out_path)
    return out_path


## 3 - AGOL helpers

In [ ]:
# ---------------------------------------------------------------------------
# AGOL helpers -- take an already-connected `gis` (this module never sees a password)
# ---------------------------------------------------------------------------
def _folder_name(f):
    if isinstance(f, dict):
        return f.get("title") or f.get("name")
    return getattr(f, "name", None) or getattr(f, "title", None)


def list_folder_titles(gis):
    me = gis.users.me
    return [n for n in (_folder_name(f) for f in (me.folders or [])) if n]


def iter_items(gis, scope, folders=None, item_type=None, max_items=2000):
    """Yield owned items. scope 'entire' = root + all folders; 'selected' = `folders`
    (use '(root)' for the root folder). item_type filters on exact Item.type."""
    me = gis.users.me
    want_type = (item_type or "").strip().lower()
    if scope == "entire":
        targets = [None] + list_folder_titles(gis)
    else:
        targets = [None if n == "(root)" else n for n in (folders or [])]
    seen, count = set(), 0
    for folder in targets:
        try:
            items = me.items(folder=folder, max_items=max_items)
        except Exception:
            items = []
        for it in items:
            if it.id in seen:
                continue
            seen.add(it.id)
            if want_type and (it.type or "").lower() != want_type:
                continue
            yield it
            count += 1
            if count >= max_items:
                return

## 4 - Run

In [ ]:
# ---- connect (no credentials in the notebook) ------------------------------
import os, getpass
from arcgis.gis import GIS
try:
    gis = GIS("home")                          # ArcGIS Notebooks / hosted
except Exception:
    try:
        gis = GIS("pro")                       # running inside ArcGIS Pro
    except Exception:
        u = os.environ.get("AGOL_USERNAME") or input("ArcGIS Online username: ")
        p = os.environ.get("AGOL_PASSWORD") or getpass.getpass("Password: ")
        gis = GIS("https://www.arcgis.com", u, p)
print("Signed in as", gis.users.me.username)

# ---- render every matching item (writes to AGOL only when DRY_RUN = False) --
rendered = updated = 0
for it in iter_items(gis, SCOPE, folders=FOLDERS or None,
                     item_type=ITEM_TYPE, max_items=MAX_ITEMS):
    png = render_thumbnail(TEMPLATE_PATH, OUTPUT_DIR, it.title, it.type, it.id,
                           brand_blue=BRAND_BLUE, max_title_chars=MAX_TITLE_CHARS,
                           max_title_lines=MAX_TITLE_LINES)
    rendered += 1
    if DRY_RUN:
        print("RENDERED:", it.title, "->", png.name)
    else:
        it.update(thumbnail=str(png)); updated += 1
        print("UPDATED:", it.title, "(", it.type, ")")
print("Done. Rendered {}{}.".format(rendered, "" if DRY_RUN else ", updated {}".format(updated)))
